In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from datasets import load_dataset
import json

# Load a pre-trained tokenizer and model (using BioBERT here)
model_name = "dmis-lab/biobert-base-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define the label set (these correspond to the entity types)
labels = ["O", "B-PATIENT_NAME", "I-PATIENT_NAME", "B-AGE", "I-AGE",
          "B-APPOINTMENT_DATE", "I-APPOINTMENT_DATE", "B-SYMPTOMS", "I-SYMPTOMS",
          "B-PAST_HISTORY", "I-PAST_HISTORY", "B-LAB_RESULTS", "I-LAB_RESULTS",
          "B-MEDICATIONS", "I-MEDICATIONS", "B-EXAMINATION", "I-EXAMINATION"]
label_to_id = {label: i for i, label in enumerate(labels)}
id_to_label = {i: label for i, label in enumerate(labels)}

# Load your dataset from the synthetic JSON Lines file
dataset = load_dataset("json", data_files="synthetic_medical_reports.jsonl", split="train")

# Function to align entities with tokens
def align_labels(example):
    text = example["report_text"]
    
    # Print a few examples of report text and annotations to debug
    print("Report Text:", text[:200])  # Print the first 200 characters of the report for readability
    print("Annotations:", example["annotation"])  # Print the entire annotation dictionary
    
    # Tokenize the text and also return offset mappings
    tokenized_input = tokenizer(text, truncation=True, padding="max_length", max_length=512, return_offsets_mapping=True)
    
    # Initialize labels with "O" (Outside, meaning not an entity)
    labels = ["O"] * len(tokenized_input["input_ids"])
    
    # For each entity in the annotation, ensure it's a string and find its position in the text
    for entity, entity_text in example["annotation"].items():
        # Convert entity_text to string if it is not already a string (to avoid the TypeError)
        entity_text = str(entity_text)
        
        # Print the entity and its text to debug
        print(f"Entity: {entity} - Text: {entity_text}")
        
        # Find the span of the entity in the text
        start_pos = text.find(entity_text)
        if start_pos == -1:
            continue  # If entity is not found in the text, skip it
        end_pos = start_pos + len(entity_text)
        
        # Loop through the tokenized input and assign labels based on the token's position
        for idx, (start, end) in enumerate(tokenized_input["offset_mapping"]):
            if start >= start_pos and end <= end_pos:
                if start == start_pos:
                    labels[idx] = f"B-{entity.upper()}"
                else:
                    labels[idx] = f"I-{entity.upper()}"
                    
    # Assign the tokenized labels
    tokenized_input["labels"] = [label_to_id[label] for label in labels]
    
    # Remove the offset mapping as it's not used for the model
    tokenized_input.pop("offset_mapping")
    
    return tokenized_input


# Apply the function to all examples in the dataset
tokenized_dataset = dataset.map(align_labels, batched=False)

# Save the tokenized dataset
tokenized_dataset.save_to_disk("tokenized_medical_reports")


c:\Users\Hardik Agrawal\anaconda3\envs\cuda_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Map:  16%|█▋        | 165/1000 [00:00<00:01, 827.56 examples/s]

Report Text: General Consultation
Patient: Steven Mendoza, Age: 53
Date: 19-06-1995

Steven Mendoza presented with fatigue, general discomfort. Past history includes depression diagnosed in college; COPD diagnosed
Annotations: {'patient_name': 'Steven Mendoza', 'age': 53, 'appointment_date': '19-06-1995', 'symptoms': 'fatigue, general discomfort', 'past_history': 'depression diagnosed in college; COPD diagnosed in 2022', 'lab_results': 'Chest CT: no consolidation; prolactin: normal', 'medications': 'Escitalopram: 10mg', 'examination': 'bleeding gums'}
Entity: patient_name - Text: Steven Mendoza
Entity: age - Text: 53
Entity: appointment_date - Text: 19-06-1995
Entity: symptoms - Text: fatigue, general discomfort
Entity: past_history - Text: depression diagnosed in college; COPD diagnosed in 2022
Entity: lab_results - Text: Chest CT: no consolidation; prolactin: normal
Entity: medications - Text: Escitalopram: 10mg
Entity: examination - Text: bleeding gums
Report Text: General Consultat

Map:  28%|██▊       | 283/1000 [00:00<00:00, 800.37 examples/s]

Report Text: General Consultation
Patient: Gina Tucker | Age: 53
Appointment Date: 01-11-1971

Detailed Report:
Gina Tucker is a 53-year-old patient presenting with snoring, insomnia, itchy skin. The patient has a
Annotations: {'patient_name': 'Gina Tucker', 'age': 53, 'appointment_date': '01-11-1971', 'symptoms': 'snoring, insomnia, itchy skin', 'past_history': 'COVID-19 in 2021', 'lab_results': 'EEG: normal', 'medications': 'Insulin aspart: 4 units', 'examination': 'reduced reflexes'}
Entity: patient_name - Text: Gina Tucker
Entity: age - Text: 53
Entity: appointment_date - Text: 01-11-1971
Entity: symptoms - Text: snoring, insomnia, itchy skin
Entity: past_history - Text: COVID-19 in 2021
Entity: lab_results - Text: EEG: normal
Entity: medications - Text: Insulin aspart: 4 units
Entity: examination - Text: reduced reflexes
Report Text: General Consultation
Patient: Natalie Perry | Age: 84
Appointment Date: 01-04-2010

Detailed Report:
Natalie Perry is a 84-year-old patient presentin

Map:  39%|███▉      | 394/1000 [00:00<00:00, 766.44 examples/s]

...

Map:  74%|███████▍  | 741/1000 [00:01<00:00, 708.81 examples/s]

Entity: patient_name - Text: Monica Ramirez
Entity: age - Text: 37
Entity: appointment_date - Text: 04-04-1976
Entity: symptoms - Text: memory loss, bad breath
Entity: past_history - Text: stroke in 2019; car accident in 2014
Entity: lab_results - Text: uric acid: high; blood ammonia: elevated
Entity: medications - Text: Nicotine patch: daily
Entity: examination - Text: pale conjunctiva
Report Text: General Consultation
Patient: Betty Simmons, Age: 20
Date: 23-04-1987

Betty Simmons presented with loss of appetite, sensitivity to sound, general discomfort. Past history includes diabetes - insulin
Annotations: {'patient_name': 'Betty Simmons', 'age': 20, 'appointment_date': '23-04-1987', 'symptoms': 'loss of appetite, sensitivity to sound, general discomfort', 'past_history': 'diabetes - insulin-dependent; bariatric surgery', 'lab_results': 'low hemoglobin; serum calcium: 9.5 mg/dL', 'medications': 'Sertraline: 50mg', 'examination': 'skin warm and dry'}
Entity: patient_name - Text: Bett

Map:  89%|████████▉ | 893/1000 [00:01<00:00, 730.26 examples/s]

Report Text: General Consultation
Patient: Betty Simmons | Age: 64
Appointment Date: 27-08-2011

Detailed Report:
Betty Simmons is a 64-year-old patient presenting with constipation. The patient has a history incl
Annotations: {'patient_name': 'Betty Simmons', 'age': 64, 'appointment_date': '27-08-2011', 'symptoms': 'constipation', 'past_history': 'organ recipient', 'lab_results': 'high LDL cholesterol', 'medications': 'Nystatin: swish and swallow', 'examination': 'positive Babinski reflex'}
Entity: patient_name - Text: Betty Simmons
Entity: age - Text: 64
Entity: appointment_date - Text: 27-08-2011
Entity: symptoms - Text: constipation
Entity: past_history - Text: organ recipient
Entity: lab_results - Text: high LDL cholesterol
Entity: medications - Text: Nystatin: swish and swallow
Entity: examination - Text: positive Babinski reflex
Report Text: General Consultation
Patient: Susan Lee, Age: 86
Date: 21-03-2007

Susan Lee presented with cold intolerance, dry mouth. Past history inclu

Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 39999.47 examples/s]


In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments
from datasets import load_from_disk

# Load the smaller model (e.g., 'bert-base-cased' or 'distilbert-base-cased')
model_name = "bert-base-cased"  # Use 'distilbert-base-cased' for an even smaller version
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(labels))

# Load your tokenized dataset (from the previous step)
tokenized_dataset = load_from_disk("tokenized_medical_reports")

# Split the dataset into training and validation sets
train_size = int(0.9 * len(tokenized_dataset))  # Use 90% of the data for training
val_size = len(tokenized_dataset) - train_size   # Use the remaining 10% for validation

train_dataset = tokenized_dataset.select(range(train_size))
val_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

# Set up the training arguments with smaller batch size and gradient accumulation
training_args = TrainingArguments(
    output_dir="./ner_output",           # Directory to save model checkpoints
    evaluation_strategy="epoch",         # Evaluate model after every epoch
    learning_rate=2e-5,                  # Learning rate
    per_device_train_batch_size=8,       # Smaller batch size
    gradient_accumulation_steps=2,       # Gradient accumulation over 2 steps
    num_train_epochs=3,                  # Number of epochs to train the model
    weight_decay=0.01,                   # Weight decay to avoid overfitting
    save_strategy="epoch",               # Save model checkpoints after every epoch
    logging_dir="./logs",                # Log directory for training logs
    logging_steps=10,                    # Log every 10 steps
    fp16=True,                           # Enable mixed precision training
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,  # Provide the validation dataset here
    tokenizer=tokenizer,
)

# Start fine-tuning the model
trainer.train()

# Save the fine-tuned model
trainer.save_model("./ner_model")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Hardik Agrawal\AppData\Local\Temp\ipykernel_15068\239784517.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 4.00 GiB of which 0 bytes is free. Of the allocated memory 6.63 GiB is allocated by PyTorch, and 96.48 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)